# QLoRA rank and preflight study

This notebook is a small planning pass before running a GPU fine-tune. It compares LoRA rank choices, estimates adapter memory, and checks how batch size plus accumulation changes the number of optimizer updates.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from qlora_lab.config import QLoRAConfig
from qlora_lab.experiments import rank_sweep_report, rank_sweep_csv_rows
from qlora_lab.preflight import build_preflight_report, estimate_update_steps

report_dir = Path("../reports/notebooks")
report_dir.mkdir(parents=True, exist_ok=True)


The default dimensions below are a rough planning profile for a small decoder-only model. They are not meant to replace the exact model config, but they are useful for comparing how rank changes trainable parameter count.

In [ ]:
rank_report = rank_sweep_report(
    ranks=[4, 8, 16, 32, 64],
    hidden_size=2048,
    intermediate_size=8192,
    layers=24,
    base_parameters=1_500_000_000,
)

rank_frame = pd.DataFrame(rank_sweep_csv_rows(rank_report))
rank_frame


In [ ]:
axis = rank_frame.plot(
    x="rank",
    y="adapter_memory_mb_fp16",
    marker="o",
    figsize=(7, 4),
    legend=False,
)
axis.set_title("LoRA rank vs adapter memory")
axis.set_xlabel("LoRA rank")
axis.set_ylabel("Adapter memory, FP16 MB")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(report_dir / "rank_memory_curve.png", dpi=160)


The next table keeps the per-device batch size small and moves the effective batch size with gradient accumulation. This is usually easier to fit on a single GPU while still getting a reasonable update cadence.

In [ ]:
batch_rows = []
for per_device_batch_size in [1, 2, 4]:
    for accumulation in [4, 8, 16]:
        config = QLoRAConfig(
            batch_size=per_device_batch_size,
            gradient_accumulation_steps=accumulation,
            epochs=1.0,
        )
        batch_rows.append(
            {
                "batch_size": per_device_batch_size,
                "grad_accumulation": accumulation,
                **estimate_update_steps(train_examples=1_000, config=config),
            }
        )

batch_frame = pd.DataFrame(batch_rows)
batch_frame


In [ ]:
planning_config = QLoRAConfig(
    lora_r=16,
    lora_alpha=32,
    batch_size=2,
    gradient_accumulation_steps=8,
    max_seq_length=512,
)

preflight = build_preflight_report(
    planning_config,
    train_examples=1_000,
    base_parameters=1_500_000_000,
)
preflight


In [ ]:
rank_frame.to_csv(report_dir / "rank_sweep_from_notebook.csv", index=False)
batch_frame.to_csv(report_dir / "batch_plan.csv", index=False)
report_dir
